<a href="https://colab.research.google.com/github/jennifer-algabre/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!git clone https://github.com/jennifer-algabre/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 158 (delta 66), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 2.01 MiB | 11.94 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/flyrank-ml-internship/flyrank-ml-internship


In [15]:
!find . -name "*.csv"

./outputs/refresh_queue_sample.csv
./data/raw/content_refresh_anonymized.csv


## 1. Unit of analysis + time window

The unit of analysis is one pseudonymized content page.

Each row represents one content page with aggregated search performance, engagement, and content metadata over the trailing 90-day observation window. The goal is to use observable search signals to identify pages that may require content review.

The starter dataset is a snapshot rather than a daily time-series dataset, so the analysis is based on aggregated historical metrics.

In [16]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

df.head()

Rows: 30,000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features
These observable signals are available before making a refresh decision and may help prioritize pages for review.

- impressions_90d
- clicks_90d
- ctr
- avg_position
- engagement_rate
- scroll_rate
- word_count
- content_age_days
- days_since_last_update
- search_volume
- competition
- content_type
- main_intent

### Label / Proxy
- Priority score (derived proxy for ranking pages)
- For exploratory analysis, trend_direction can also be used to identify declining pages, but it is not used as an input feature.

### Context
These fields help identify or describe records but are not used for prediction.

- content_id
- client_id
- impression_tier
- position_tier
- competition_level

### Excluded
The following fields are excluded because they leak information about the outcome or are not appropriate as predictive features.

- trend_direction — represents the observed outcome.
- trend_pct — directly measures the change used to determine trend direction and would introduce data leakage.
- ai_traffic_pct — not central to the current lane and may not generalize.

## 3. Verify it with queries (grain, counts, missing values, windows)



The following queries verify the main assumptions of the data contract.

**Query 1** confirms the size of the starter dataset.

**Query 2** verifies the grain by showing that each row represents one unique content page.

**Query 3** checks feature availability by counting non-missing values for each selected feature.

The label distribution is also displayed to understand the balance of the prediction target before modeling.

In [17]:
# Query 1: Dataset size

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 30,000
Columns: 44


In [18]:
# Query 2: Verify one row represents one content page

print("Unique content IDs:", df["content_id"].nunique())
print("Total rows:", len(df))
print("One row per content page:",
      df["content_id"].nunique() == len(df))

Unique content IDs: 30000
Total rows: 30000
One row per content page: True


In [19]:
# Query 3: Verify feature availability

availability = (
    df[feature_cols]
    .notna()
    .sum()
    .rename("Available Rows")
    .to_frame()
)

availability["Availability (%)"] = (
    availability["Available Rows"] / len(df) * 100
).round(2)

availability.sort_values("Availability (%)")

,Available Rows,Availability (%)
word_count,22301,74.34
search_volume,27532,91.77
competition,27532,91.77
main_intent,27626,92.09
scroll_rate,29875,99.58
avg_position,30000,100.00
impressions_90d,30000,100.00
clicks_90d,30000,100.00
content_age_days,30000,100.00
engagement_rate,30000,100.00


In [20]:
# Label distribution

df["trend_direction"].value_counts()

,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


## 4. Data limits

This starter dataset is intended for learning and model development, but it has several limitations.

Each row is an aggregated snapshot rather than daily performance data, so it cannot show exactly when changes in search performance occurred. The dataset also contains only pseudonymized content and client IDs, which cannot be linked back to real websites or business context.

Some columns require careful interpretation. For example, `avg_position = 0` represents missing data rather than a true ranking, and several features such as `word_count` contain missing values that require preprocessing. Rate columns (such as `ctr`, `engagement_rate`, and `scroll_rate`) are stored as percentages on a 0–100 scale and should be interpreted accordingly.

Finally, the label (`trend_direction`) is derived from `trend_pct`, so those fields must not be used together as model features because they would introduce data leakage. The results of this project should therefore be interpreted as decision support based on observed search performance signals rather than evidence of causal relationships.